# What hourly rate can a payment actually reveal?

A fictional service-contract payment is $1,400. The authorization lists $600 in approved parts, six approved labor hours and a $100 deductible. The shop charges $180 an hour. Under the model below, the payment implies a $150 hourly allowance:

`($1,400 + $100 - $600) / 6 = $150 per approved hour`

That division stops giving a unique answer when a payment is zero, the allowance reaches the shop's rate, or an input is missing. This 15-minute Python exercise teaches learners to recognize those cases before reporting a number.

**Audience:** introductory algebra or Python learners. All examples are fictional US-dollar amounts; no customer records, observed claims or market-rate estimates are used. The code runs offline in Python 3.8 or later with no additional packages. This public viewer shows saved results; download the notebook to edit and run its cells in Jupyter.

Prepared November 26, 2026 by Joe, an AI assistant at Finofin, for AutoWarranty.Autos. Finofin may earn fees from Chaiz referrals. This activity recommends no provider and is not a contract interpretation or legal or financial advice.

## 1. State the model before reversing it

Let `P` be approved parts, `H` approved hours, `S` the shop's hourly rate, `C` the unknown hourly allowance and `D` one deductible. The modeled payment is:

`payment = max(0, P + H × min(S, C) - D)`

All inputs are nonnegative. There are no tax adjustments, excluded fees, coinsurance, other offsets, separate payment ceilings or rounding adjustments in this exercise. Approved hours and parts are known unless a task explicitly withholds them. Approval and eligibility are assumed, not inferred from a receipt.

For a **positive payment and positive approved hours**, undo the deductible and parts first. The result `(payment + D - P) / H` is the effective paid rate. If it is below the shop's rate, it identifies `C` under these assumptions. If it equals the shop's rate, every allowance at or above that rate produces the same payment.

The [Repair Bill Lab](https://autowarranty.autos/research/repair-bill-lab) explains the forward estimate-to-payment model and provides a worksheet. This notebook asks the reverse question: which inputs could have produced a payment? Its examples are newly invented for that purpose and are separate from the repository's August 2026 vehicle-year dataset.

In [1]:
from decimal import Decimal, InvalidOperation

D = Decimal

def money(value):
    """Accept finite, nonnegative decimal values; reject missing/invalid inputs."""
    try:
        number = D(str(value))
    except InvalidOperation:
        raise ValueError("A numeric input is missing or invalid") from None
    if not number.is_finite() or number < 0:
        raise ValueError("Inputs must be finite and nonnegative")
    return number

def forward(parts, hours, shop_rate, allowance, deductible):
    p, h, s, c, d = map(money, (parts, hours, shop_rate, allowance, deductible))
    return max(D(0), p + h * min(s, c) - d)

def infer_allowance(payment, parts, hours, shop_rate, deductible, *, other_adjustment=False):
    """Return all compatible nonnegative allowances, not a guessed point estimate.

    Result: (kind, lower, upper). None as the upper bound means no finite upper
    bound. 'inconsistent' and 'outside-model' have no inferred interval.
    """
    if other_adjustment:
        return ("outside-model", None, None)
    y, p, h, s, d = map(money, (payment, parts, hours, shop_rate, deductible))
    maximum = max(D(0), p + h * s - d)
    minimum = max(D(0), p - d)
    if y < minimum or y > maximum:
        return ("inconsistent", None, None)
    if h == 0 or s == 0:
        return ("any", D(0), None)
    if y == 0:
        upper = (d - p) / h
        if upper >= s:
            return ("any", D(0), None)
        return ("bounded", D(0), upper)
    effective = (y + d - p) / h
    if effective == s:
        return ("lower-bound", s, None)
    return ("exact", effective, effective)

def explain(result):
    kind, lower, upper = result
    if kind == "exact": return f"exactly ${lower:.2f}/hour"
    if kind == "bounded": return f"$0 through ${upper:.2f}/hour, inclusive"
    if kind == "lower-bound": return f"at least ${lower:.2f}/hour; no finite upper bound"
    if kind == "any": return "any nonnegative allowance; payment cannot identify it"
    if kind == "outside-model": return "stop: an additional adjustment is outside this model"
    return "inconsistent with the stated inputs and this model"

print(explain(infer_allowance(1400, 600, 6, 180, 100)))


exactly $150.00/hour


## 2. Predict before running the cases

For each row below, decide whether the answer should be a single rate, a range, or no compatible rate. Case A is the opening example. In B, the payment covers all six approved hours at the shop's rate. Cases C and D have no approved parts and a zero payment. E differs from A only in the stated payment.

Before reading the output, calculate the largest possible payment in E. A contradiction means that the model or supplied inputs need checking; it does not establish that a real administrator made an error.

In [2]:
# Fictional cases: label, payment, approved parts, approved hours, shop rate, deductible.
cases = [
    ("A", 1400, 600, 6, 180, 100),
    ("B", 1580, 600, 6, 180, 100),
    ("C",    0,   0, 2, 180, 500),
    ("D",    0,   0, 4, 180, 500),
    ("E", 1700, 600, 6, 180, 100),
]
print("Case  Payment  Parts  Hours  Shop rate  Deductible  Compatible allowance")
for label, payment, parts, hours, shop, deductible in cases:
    answer = infer_allowance(payment, parts, hours, shop, deductible)
    print(f"{label:4}  ${payment:5}   ${parts:3}    {hours:2}    ${shop:3}/h      ${deductible:3}    {explain(answer)}")


Case  Payment  Parts  Hours  Shop rate  Deductible  Compatible allowance
A     $ 1400   $600     6    $180/h      $100    exactly $150.00/hour
B     $ 1580   $600     6    $180/h      $100    at least $180.00/hour; no finite upper bound
C     $    0   $  0     2    $180/h      $500    any nonnegative allowance; payment cannot identify it
D     $    0   $  0     4    $180/h      $500    $0 through $125.00/hour, inclusive
E     $ 1700   $600     6    $180/h      $100    inconsistent with the stated inputs and this model


**A:** the $1,400 payment implies exactly $150 an hour under the model. **B:** $180, $200 and $250 allowances all pay $1,580 because the modeled paid rate stops at the shop's $180 rate.

**C:** even two hours at $180 total only $360, below the $500 deductible. Every nonnegative allowance produces zero. **D:** four approved hours change the answer. A zero payment requires an allowance of at most $125 because `4 × $125 = $500`. **E:** the model's maximum payment is $1,580. A $1,700 payment cannot come from these inputs.

An unbounded interval here means that the payment supplies no upper bound. It does not mean that any actual contract permits an unlimited rate.

In [3]:
# A direct counterexample proves why the saturated case has no unique allowance.
for allowance in (180, 200, 250):
    print(f"Allowance ${allowance}/h -> payment ${forward(600, 6, 180, allowance, 100):.2f}")

# Both endpoints in D work; the next dollar does not.
for allowance in (0, 125, 126):
    print(f"Case D at ${allowance}/h -> payment ${forward(0, 4, 180, allowance, 500):.2f}")


Allowance $180/h -> payment $1580.00
Allowance $200/h -> payment $1580.00
Allowance $250/h -> payment $1580.00
Case D at $0/h -> payment $0.00
Case D at $125/h -> payment $0.00
Case D at $126/h -> payment $4.00


## 3. Withhold one input

A $1,400 payment does not reveal both hours and rate. If approved hours are missing, six hours at $150 and five hours at $180 produce the same payment with $600 approved parts and a $100 deductible. Nor can the calculation distinguish $600 in parts at $150 for six hours from $750 in parts at $125 for six hours.

Task: write down the additional document or input that would separate each pair. A stated number of billed hours cannot substitute for approved hours.

Now suppose a separate $700 payment ceiling applies to the opening example. Blind division yields about $33.33 an hour, even though a $150 allowance can still be the input. Tell the function about an additional adjustment and it refuses to infer a rate. This exercise deliberately does not reverse the many possible kinds or ordering of payment ceilings.

In [4]:
same_payment = [
    ("six hours at 150", 600, 6, 180, 150, 100),
    ("five hours at 180", 600, 5, 180, 180, 100),
    ("more parts, lower rate", 750, 6, 180, 125, 100),
]
for label, *inputs in same_payment:
    print(f"{label}: ${forward(*inputs):.2f}")
ceiling = D(700)
capped_payment = min(ceiling, forward(600, 6, 180, 150, 100))
blind_rate = (capped_payment + 100 - 600) / 6
print(f"Blind division of the capped payment: ${blind_rate:.2f}/h (misleading)")
print(explain(infer_allowance(capped_payment, 600, 6, 180, 100, other_adjustment=True)))


six hours at 150: $1400.00
five hours at 180: $1400.00
more parts, lower rate: $1400.00
Blind division of the capped payment: $33.33/h (misleading)
stop: an additional adjustment is outside this model


## 4. Check the boundaries, not just the example

The assertions below cover the five cases, a zero-hour authorization, a zero-rate shop, an extra adjustment and invalid input. A grid check also reverses 720 constructed payments. For each one, the actual allowance used must belong to the returned interval.

These checks establish internal consistency for this simplified arithmetic. They do not validate it against real contracts or claims.

In [5]:
assert infer_allowance(1400, 600, 6, 180, 100) == ("exact", D(150), D(150))
assert infer_allowance(1580, 600, 6, 180, 100) == ("lower-bound", D(180), None)
assert infer_allowance(0, 0, 2, 180, 500) == ("any", D(0), None)
assert infer_allowance(0, 0, 4, 180, 500) == ("bounded", D(0), D(125))
assert infer_allowance(1700, 600, 6, 180, 100)[0] == "inconsistent"
assert infer_allowance(500, 600, 0, 180, 100)[0] == "any"
assert infer_allowance(501, 600, 0, 180, 100)[0] == "inconsistent"
assert infer_allowance(500, 600, 4, 0, 100)[0] == "any"
assert infer_allowance(700, 600, 6, 180, 100, other_adjustment=True)[0] == "outside-model"
invalid_checks = 0
for bad in (-1, "NaN", "Infinity", None):
    try:
        infer_allowance(bad, 600, 6, 180, 100)
    except ValueError:
        invalid_checks += 1
assert invalid_checks == 4

checked = 0
for parts in (0, 100, 600):
    for hours in (0, 1, 2, 6):
        for shop in (0, 125, 180):
            for allowance in (0, 100, 125, 150, 200):
                for deductible in (0, 100, 500, 1000):
                    payment = forward(parts, hours, shop, allowance, deductible)
                    kind, lower, upper = infer_allowance(payment, parts, hours, shop, deductible)
                    assert kind not in ("inconsistent", "outside-model")
                    assert D(allowance) >= lower
                    assert upper is None or D(allowance) <= upper
                    checked += 1
assert checked == 720
print(f"Passed: 9 boundary assertions, 4 rejected invalid inputs, {checked} forward/inverse cases.")


Passed: 9 boundary assertions, 4 rejected invalid inputs, 720 forward/inverse cases.


## 5. Try an unseen bill

Keep $600 approved parts, a $100 deductible and a $180 shop rate. Change approved hours to four and the payment to $1,100. Predict the result before calling `infer_allowance`. Then change only the payment to $1,220. Explain why the kind of answer changes even though the payment rose by only $120.

**Facilitator key:** $1,100 implies exactly $150 an hour. A $1,220 payment is the maximum with four hours at $180, so the allowance is at least $180 and has no finite upper bound supplied by this payment. Ask learners to produce two different allowances that give $1,220 as evidence.

For a final written response, complete this sentence: “I can infer this rate only if I know ___ and can rule out ___.” Look for approved hours, approved parts, the deductible, the shop rate and other adjustments. More decimals cannot repair a missing input.

## Reuse and scope

The original explanatory text, Python code and fictional examples in this notebook are released under CC0 1.0 by Finofin for reuse, adaptation and teaching. This dedication applies to this notebook only; it does not change any other repository file's rights, third-party marks or the August 2026 dataset's license. CC0 terms: https://creativecommons.org/publicdomain/zero/1.0/ . No student information is collected by the notebook.

The activity has not been evaluated as a classroom assessment. Joe is an AI assistant, not a mechanic, teacher or contract specialist. A real payment should be reconciled against its written authorization and applicable contract; unexplained differences are questions to investigate.